# 📘 CIFAR-10 Image Classification Learning Project
## Build and Compare **ANN vs CNN** on CIFAR-10

This notebook is designed for **students and beginners** to learn:
- How image classification works
- Why **CNN performs better than ANN**
- How architecture impacts performance
- How training strategies improve results

🎯 **Learning Goal:** Understand the complete DL pipeline by **reading the markdown + running the ready code**.

# 🧠 Problem Statement
Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare:
- Accuracy
- Loss curves
- Generalization
- Training strategies (dropout, batch norm, augmentation)

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)
print('Keras version:', tf.keras.__version__)

ModuleNotFoundError: No module named 'tensorflow'

# 📥 Load Dataset
We use **CIFAR-10**, which contains **60,000 color images of size 32×32×3**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('Train images shape:', x_train.shape)  # (50000, 32, 32, 3)
print('Test images shape :', x_test.shape)   # (10000, 32, 32, 3)
print('Train labels shape:', y_train.shape)
print('Pixel value range  :', x_train.min(), '-', x_train.max())

## 🖼️ Visualize Sample Images

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]], fontsize=10)
    plt.axis('off')
plt.suptitle('Sample CIFAR-10 Images', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# 🧹 Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

In [ ]:
# Normalize pixel values: 0-255 → 0.0-1.0
x_train_norm = x_train / 255.0
x_test_norm  = x_test  / 255.0

# ANN requires flat input: (50000, 32*32*3) = (50000, 3072)
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat  = x_test_norm.reshape(len(x_test_norm),  -1)

print('CNN input shape (normalized):', x_train_norm.shape)
print('ANN input shape (flattened) :', x_train_flat.shape)

# 🔹 Part 1: ANN Model
ANN treats images as **flat vectors**, so it cannot preserve spatial features.
This helps students understand **why CNN is better for images**.

In [ ]:
# Build ANN model
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
], name='ANN')

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

# Train ANN
ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f'ANN  →  Test Loss: {ann_test_loss:.4f}  |  Test Accuracy: {ann_test_acc*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ann_history.history['accuracy'],     label='Train Acc', color='steelblue')
axes[0].plot(ann_history.history['val_accuracy'], label='Val Acc',   color='tomato')
axes[0].set_title('ANN — Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ann_history.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(ann_history.history['val_loss'], label='Val Loss',   color='tomato')
axes[1].set_title('ANN — Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('ANN Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

# 🔹 Part 2: CNN Model
CNN preserves **spatial relationships** using:
- Convolution layers
- Pooling
- Feature extraction
- Hierarchical learning

This is why CNN performs much better for image tasks.

In [ ]:
# Build CNN model
cnn_model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    # Classifier head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

# Train CNN
cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f'CNN  →  Test Loss: {cnn_test_loss:.4f}  |  Test Accuracy: {cnn_test_acc*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cnn_history.history['accuracy'],     label='Train Acc', color='steelblue')
axes[0].plot(cnn_history.history['val_accuracy'], label='Val Acc',   color='tomato')
axes[0].set_title('CNN — Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(cnn_history.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(cnn_history.history['val_loss'], label='Val Loss',   color='tomato')
axes[1].set_title('CNN — Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

## 📈 Compare Learning Curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(ann_history.history['val_accuracy'], label='ANN Val Acc',  color='gray',     linestyle='--')
plt.plot(cnn_history.history['val_accuracy'], label='CNN Val Acc',  color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('ANN vs CNN — Validation Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(ann_history.history['val_loss'], label='ANN Val Loss', color='gray',     linestyle='--')
plt.plot(cnn_history.history['val_loss'], label='CNN Val Loss', color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('ANN vs CNN — Validation Loss')
plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 🚀 Training Strategy Upgrade: Data Augmentation
This strategy improves generalization by generating transformed images.

In [ ]:
# Data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
], name='augmentation')

# CNN with augmentation built into the model
aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_with_Augmentation')

aug_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_cnn_model.summary()

# Train Augmented CNN
aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f'Aug CNN → Test Loss: {aug_test_loss:.4f}  |  Test Accuracy: {aug_test_acc*100:.2f}%')

# 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Model'         : ['ANN', 'CNN', 'CNN + Augmentation'],
    'Parameters'    : [ann_model.count_params(), cnn_model.count_params(), aug_cnn_model.count_params()],
    'Test Accuracy' : [round(ann_test_acc*100, 2), round(cnn_test_acc*100, 2), round(aug_test_acc*100, 2)],
    'Test Loss'     : [round(ann_test_loss, 4),    round(cnn_test_loss, 4),    round(aug_test_loss, 4)]
})
print(comparison.to_string(index=False))

In [ ]:
models_list = ['ANN', 'CNN', 'CNN + Aug']
accs = [ann_test_acc * 100, cnn_test_acc * 100, aug_test_acc * 100]
colors = ['#888780', '#3266ad', '#1D9E75']

plt.figure(figsize=(7, 4))
bars = plt.bar(models_list, accs, color=colors, edgecolor='white', width=0.5)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 100)
plt.ylabel('Test Accuracy (%)')
plt.title('Model Comparison — CIFAR-10 Test Accuracy')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 🎓 Student Learning Tasks
Try these tasks after understanding the notebook:

### ✅ Beginner Tasks
1. Increase ANN layers and observe performance
2. Change CNN filters from 32→64→128
3. Increase epochs to 20
4. Add **EarlyStopping**
5. Add **data augmentation training**

# ✅ Conclusion
- **ANN works**, but ignores image structure
- **CNN extracts spatial features**, so it performs significantly better
- **Training strategies** like dropout, batch norm, and augmentation improve results
- This project builds strong fundamentals for **computer vision interviews and deep learning projects**